## Date Understanding

## Dataset Column Reference

| Column | Type | Description |
|--------|------|-------------|
| `created_date` | datetime | When the complaint was submitted by the citizen |
| `closed_date` | datetime | When the complaint was officially closed — NaT means still open |
| `agency_name` | str | Full name of the city agency responsible for handling the complaint |
| `complaint_type` | str | Main category of the complaint (e.g. Noise, Illegal Parking, HEAT/HOT WATER) |
| `complaint_detail` | str | More specific description of the complaint — filled with 'No Detail' where missing |
| `incident_zip` | str | ZIP code where the incident occurred |
| `street_name` | str | Name of the street where the incident occurred |
| `status` | str | Current state of the request (Closed, Open, Pending) |
| `resolution_description` | str | How the complaint was resolved — filled with 'No Resolution Yet' for open complaints |
| `resolution_action_updated_date` | datetime | Last date any resolution action was recorded — NaT means no action taken yet |
| `police_precinct` | str | NYPD precinct covering the incident location |
| `borough` | str | NYC borough where the incident occurred (BROOKLYN, QUEENS, MANHATTAN, BRONX, STATEN ISLAND) |
| `open_data_channel_type` | str | How the complaint was submitted (PHONE, MOBILE, ONLINE, etc.) |
| `latitude` | float | Geographic latitude coordinate of the incident |
| `longitude` | float | Geographic longitude coordinate of the incident |

In [40]:
# Import the required libraries

import numpy as np , pandas as pd , plotly.express as px
from geopy.geocoders import Nominatim
import time
# from datasist.structdata import detect_outliers 

In [41]:
# Load the Cleaned Version of Data

df = pd.read_parquet('nyc_cleaned.parquet')
df


,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,police_precinct,borough,open_data_channel_type,latitude,longitude
0,2023-06-01 01:04:22,2023-06-07 11:53:03,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,2023-06-07 00:00:00,Precinct 25,MANHATTAN,PHONE,40.797065,-73.933855
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,Precinct 52,BRONX,MOBILE,40.860566,-73.913341
2,2023-06-01 01:03:51,2023-06-01 01:35:53,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,2023-06-01 01:35:58,Precinct 66,BROOKLYN,PHONE,40.624388,-73.970522
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,Precinct 71,BROOKLYN,MOBILE,40.663251,-73.936208
4,2023-06-01 01:03:31,2023-06-01 01:12:32,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,2023-06-01 01:12:37,Precinct 9,MANHATTAN,ONLINE,40.724409,-73.978610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3129047,2022-06-01 01:08:15,2022-06-01 01:40:44,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10024,WEST 90 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:40:47,Precinct 24,MANHATTAN,MOBILE,40.789428,-73.971178
3129048,2022-06-01 01:07:59,2022-06-01 03:43:09,New York City Police Department,Noise - Commercial,Loud Talking,10466,WHITE PLAINS ROAD,Closed,The Police Department responded to the complai...,2022-06-01 03:43:13,Precinct 47,BRONX,MOBILE,40.896700,-73.855539
3129049,2022-06-01 01:06:46,2022-06-02 11:00:32,New York City Police Department,Noise - Residential,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,2022-06-02 11:01:13,Precinct 47,BRONX,MOBILE,40.892385,-73.859216
3129050,2022-06-01 01:06:42,2022-06-01 01:20:07,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,11236,EAST 108 STREET,Closed,The Police Department responded to the complai...,2022-06-01 01:20:14,Precinct 69,BROOKLYN,ONLINE,40.651047,-73.894917


### Data Exploation

Check for :
- Types
- NANs 
- Duplicated Values 
- Summary Statistics for (Numerical & Categorical) 

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 2930915 entries, 0 to 3129051
Data columns (total 15 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   created_date                    datetime64[us]
 1   closed_date                     datetime64[us]
 2   agency_name                     str           
 3   complaint_type                  str           
 4   complaint_detail                str           
 5   incident_zip                    str           
 6   street_name                     str           
 7   status                          str           
 8   resolution_description          str           
 9   resolution_action_updated_date  datetime64[us]
 10  police_precinct                 str           
 11  borough                         str           
 12  open_data_channel_type          str           
 13  latitude                        float64       
 14  longitude                       float64       
dtypes: datetime64[

### Note: 
- It's normal to still have NANs in both (closed_date & resolution_action_updated_date), it just refers to that the complaint not solved yet, So keeping these NaNs make sense.
- In this case the status stayed open as it.

In [43]:
df.isna().sum()

created_date                          0
closed_date                       26531
agency_name                           0
complaint_type                        0
complaint_detail                      0
incident_zip                          0
street_name                           0
status                                0
resolution_description                0
resolution_action_updated_date    10838
police_precinct                       0
borough                               0
open_data_channel_type                0
latitude                              0
longitude                             0
dtype: int64

In [44]:
# Check Duplicates

df.duplicated().sum()

np.int64(0)

In [45]:
# Check Summary Statitics for Numerical Columns
df.describe()

,created_date,closed_date,resolution_action_updated_date,latitude,longitude
count,2930915,2904384,2920077,2.930915e+06,2.930915e+06
mean,2022-11-27 17:16:54.266789,2022-12-24 04:20:47.431534,2022-12-24 06:56:29.103417,4.073873e+01,-7.392220e+01
min,2022-06-01 01:05:52,2020-09-24 00:00:00,2020-09-24 00:00:00,4.049854e+01,-7.425490e+01
25%,2022-08-26 21:51:25.500000,2022-09-06 14:05:27,2022-09-06 00:00:00,4.067329e+01,-7.396528e+01
50%,2022-11-23 17:42:32,2022-12-10 05:48:17,2022-12-10 00:48:22,4.073216e+01,-7.392472e+01
75%,2023-03-01 13:10:52,2023-03-19 23:12:52,2023-03-19 23:10:33,4.082044e+01,-7.387422e+01
max,2023-06-01 01:04:22,2026-04-17 14:41:43,2026-04-17 14:41:47,4.091289e+01,-7.370038e+01
std,NaN,NaN,NaN,8.802597e-02,7.678091e-02


In [46]:
# Check Summary statistics for Categorical Columns

df.describe(include= 'str')


,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,police_precinct,borough,open_data_channel_type
count,2930915,2930915,2930915,2930915,2930915,2930915,2930915,2930915,2930915,2930915
unique,14,188,902,241,11407,7,748,78,5,5
top,New York City Police Department,Illegal Parking,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,Precinct 47,BROOKLYN,ONLINE
freq,1368289,433372,416392,86854,57348,2900053,386421,121157,882842,1367924


### Data Cleaning

- In depth check for the distribution of numerical and (n)Unique for Categorical. 
- Trying to find the borough with 'Unspecified' value handled by lat/long using geopy reverse method.

In [47]:
num_cols = df.select_dtypes(include='number')
num_cols # {latitude,longitude} are a mapping cloumns, so its' distribution itself doesn't reflects any useful right now!

,latitude,longitude
0,40.797065,-73.933855
1,40.860566,-73.913341
2,40.624388,-73.970522
3,40.663251,-73.936208
4,40.724409,-73.978610
...,...,...
3129047,40.789428,-73.971178
3129048,40.896700,-73.855539
3129049,40.892385,-73.859216
3129050,40.651047,-73.894917


In [48]:
cat_cols = df.select_dtypes(include='str')
cat_cols

,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,police_precinct,borough,open_data_channel_type
0,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,Precinct 25,MANHATTAN,PHONE
1,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,Precinct 52,BRONX,MOBILE
2,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,Precinct 66,BROOKLYN,PHONE
3,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,Precinct 71,BROOKLYN,MOBILE
4,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,Precinct 9,MANHATTAN,ONLINE
...,...,...,...,...,...,...,...,...,...,...
3129047,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,10024,WEST 90 STREET,Closed,The Police Department responded to the complai...,Precinct 24,MANHATTAN,MOBILE
3129048,New York City Police Department,Noise - Commercial,Loud Talking,10466,WHITE PLAINS ROAD,Closed,The Police Department responded to the complai...,Precinct 47,BRONX,MOBILE
3129049,New York City Police Department,Noise - Residential,Loud Music/Party,10466,EAST 231 STREET,Closed,The Police Department responded to the complai...,Precinct 47,BRONX,MOBILE
3129050,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,11236,EAST 108 STREET,Closed,The Police Department responded to the complai...,Precinct 69,BROOKLYN,ONLINE


In [49]:
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

agency_name
14
<ArrowStringArray>
['Department of Housing Preservation and Development',
                    'New York City Police Department',
                    'Department of Homeless Services',
             'Department of Environmental Protection',
                       'Department of Transportation',
       'Department of Consumer and Worker Protection',
                           'Department of Sanitation',
                 'Department of Parks and Recreation',
            'Department of Health and Mental Hygiene',
                      'Taxi and Limousine Commission',
                            'Department of Buildings',
                   'Economic Development Corporation',
                            'Department of Education',
                'Office of Technology and Innovation']
Length: 14, dtype: str
----------------------------------------------------------------------------------------------------
complaint_type
188
<ArrowStringArray>
[                                 

In [50]:
df.head()

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,police_precinct,borough,open_data_channel_type,latitude,longitude
0,2023-06-01 01:04:22,2023-06-07 11:53:03,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,2023-06-07 00:00:00,Precinct 25,MANHATTAN,PHONE,40.797065,-73.933855
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,Precinct 52,BRONX,MOBILE,40.860566,-73.913341
2,2023-06-01 01:03:51,2023-06-01 01:35:53,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,2023-06-01 01:35:58,Precinct 66,BROOKLYN,PHONE,40.624388,-73.970522
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,Precinct 71,BROOKLYN,MOBILE,40.663251,-73.936208
4,2023-06-01 01:03:31,2023-06-01 01:12:32,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,2023-06-01 01:12:37,Precinct 9,MANHATTAN,ONLINE,40.724409,-73.978610


In [51]:
df['borough'].value_counts()

borough
BROOKLYN         882842
QUEENS           679385
BRONX            650416
MANHATTAN        599406
STATEN ISLAND    118866
Name: count, dtype: int64

In [52]:
df[df['borough'] == 'Unspecified']

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,police_precinct,borough,open_data_channel_type,latitude,longitude


In [53]:
# Apply on one coordinate 
geolocators = Nominatim(user_agent= 'Real Borough')
location = geolocators.reverse('40.704748,-73.914030')
location.raw['address']['suburb']

'Queens'

In [54]:
# Apply this for all Unspecified Values:

def get_borough(lat, lon):
    try:
        time.sleep(1) 
        location = geolocators.reverse((lat, lon), timeout=10)
        address = location.raw['address']
        
        suburb = address.get('suburb', '')
        
        borough_map = {
            'Queens'       : 'QUEENS',
            'Brooklyn'     : 'BROOKLYN',
            'Manhattan'    : 'MANHATTAN',
            'Bronx'        : 'BRONX',
            'Staten Island': 'STATEN ISLAND'
        }
        
        for key, value in borough_map.items():
            if key in suburb:
                return value
                
        return 'Unspecified'  
    
    except:
        return 'Unspecified'  
    


In [55]:
mask = df['borough'] == 'Unspecified'
df.loc[mask, 'borough'] = df[mask].apply(
    lambda row: get_borough(row['latitude'], row['longitude']), axis=1
)

In [56]:
print(df['borough'].value_counts())

borough
BROOKLYN         882842
QUEENS           679385
BRONX            650416
MANHATTAN        599406
STATEN ISLAND    118866
Name: count, dtype: int64


In [57]:
df.to_parquet('nyc_cleaned.parquet', compression='snappy')

### Feature Engineering

- Respone time columns
- Extract date periods to get top complaints in[days, month] 
- Extract day periods[hour] --> [morning, evening,..]

In [58]:
df['response_time_hours'] = (
    df['closed_date'] - df['created_date']
).dt.total_seconds() / 3600
df['response_time_hours']

0          154.811389
1            0.542222
2            0.533889
3            0.107778
4            0.150278
              ...    
3129047      0.541389
3129048      2.586111
3129049     33.896111
3129050      0.223611
3129051      2.875833
Name: response_time_hours, Length: 2930915, dtype: float64

In [59]:
df['response_time_days'] = df['response_time_hours'] // 24
df['response_time_days']

0          6.0
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
3129047    0.0
3129048    0.0
3129049    1.0
3129050    0.0
3129051    0.0
Name: response_time_days, Length: 2930915, dtype: float64

In [67]:
df['complaint_day'] = df['created_date'].dt.day_name()
df['complaint_hour'] = df['created_date'].dt.hour
df['complaint_month'] = df['created_date'].dt.month_name()

In [68]:
# Create month column to be extracted with the year 
df['complaint_month_year'] = df['created_date'].dt.to_period('M').dt.to_timestamp()
df['complaint_month_year']

0         2023-06-01
1         2023-06-01
2         2023-06-01
3         2023-06-01
4         2023-06-01
             ...    
3129047   2022-06-01
3129048   2022-06-01
3129049   2022-06-01
3129050   2022-06-01
3129051   2022-06-01
Name: complaint_month_year, Length: 2930915, dtype: datetime64[us]

In [62]:
# Period of day derived from hour

def get_period(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['complaint_period'] = df['complaint_hour'].apply(get_period)

In [70]:
df.head()

,created_date,closed_date,agency_name,complaint_type,complaint_detail,incident_zip,street_name,status,resolution_description,resolution_action_updated_date,...,open_data_channel_type,latitude,longitude,response_time_hours,response_time_days,complaint_day,complaint_hour,complaint_period,complaint_month,complaint_month_year
0,2023-06-01 01:04:22,2023-06-07 11:53:03,Department of Housing Preservation and Develop...,PLUMBING,BASIN/SINK,10035,EAST 118 STREET,Closed,The Department of Housing Preservation and Dev...,2023-06-07 00:00:00,...,PHONE,40.797065,-73.933855,154.811389,6.0,Thursday,1,Night,June,2023-06-01
1,2023-06-01 01:04:11,2023-06-01 01:36:43,New York City Police Department,Illegal Parking,Blocked Hydrant,10468,CEDAR AVENUE,Closed,The Police Department reviewed your complaint ...,2023-06-01 01:36:48,...,MOBILE,40.860566,-73.913341,0.542222,0.0,Thursday,1,Night,June,2023-06-01
2,2023-06-01 01:03:51,2023-06-01 01:35:53,New York City Police Department,Illegal Parking,Blocked Sidewalk,11230,OCEAN PARKWAY,Closed,The Police Department responded and upon arriv...,2023-06-01 01:35:58,...,PHONE,40.624388,-73.970522,0.533889,0.0,Thursday,1,Night,June,2023-06-01
3,2023-06-01 01:03:49,2023-06-01 01:10:17,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,11203,LEFFERTS AVENUE,Closed,The Police Department responded and upon arriv...,2023-06-01 01:10:22,...,MOBILE,40.663251,-73.936208,0.107778,0.0,Thursday,1,Night,June,2023-06-01
4,2023-06-01 01:03:31,2023-06-01 01:12:32,New York City Police Department,Noise - Residential,Loud Music/Party,10009,AVENUE C,Closed,The Police Department responded to the complai...,2023-06-01 01:12:37,...,ONLINE,40.724409,-73.978610,0.150278,0.0,Thursday,1,Night,June,2023-06-01


In [71]:
# Save the last version
df.to_parquet('nyc_featured.parquet' , compression= 'snappy')